In [1]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
    
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.preprocessing import PowerTransformer

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [7]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.00 0.95      0.08
SUVpeak                   0.29 0.59      0.76
TLG                       0.22 0.64      0.65
age                       1.60 0.21      2.28
cavum_oris                0.00 1.00      0.01
charlson                  0.07 0.79      0.34
female                    0.17 0.68      0.56
histgrade_high            1.26 0.26      1.93
hpv_related               4.31 0.04      4.72
hypopharynx               0.00 0.99      0.01
larynx                    0.00 0.98      0.02
oropharynx                0.00 0.99      0.02
pack_years                0.71 0.40      1.32
uicc8_III-IV              0.45 0.50      0.99


In [8]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [9]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.02 0.88      0.18
SUVpeak                   0.56 0.46      1.13
TLG                       0.06 0.81      0.30
age                       1.58 0.21      2.26
cavum_oris                0.08 0.77      0.37
charlson                  0.03 0.86      0.22
female                    0.35 0.56      0.85
histgrade_high            0.95 0.33      1.60
hpv_related               3.02 0.08      3.60
hypopharynx               0.10 0.75      0.41
larynx                    1.46 0.23      2.14
oropharynx                0.96 0.33      1.61
pack_years                0.39 0.53      0.92
uicc8_III-IV              0.08 0.78      0.36


In [10]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [18]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [22]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Yeo-Johnson Transformation

In [23]:
original_X = X.copy()

In [24]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
pt = PowerTransformer(method='yeo-johnson')
X_numeric_std = pt.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [28]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [29]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [30]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [31]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.785036,1,0,1,0,0,1,0.0,0,-1.533345,0.0,0.750782,0.080033,0.343545
1,-0.746998,0,0,0,0,1,0,0.0,1,0.397233,0.0,-1.213660,-1.722060,-1.657833
2,-0.175256,0,1,0,0,0,1,0.0,1,0.830437,1.0,-0.474112,0.735716,0.318631
3,1.371631,0,0,0,0,1,0,0.0,1,0.727938,0.0,-1.992620,-1.291020,-1.835650
4,0.987008,0,0,0,0,1,0,0.0,1,1.144253,0.0,-1.106304,-0.816679,-1.003396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.007949,0,0,1,0,0,1,1.0,0,-1.533345,0.0,-0.158214,-0.822461,-0.611074
135,1.111443,0,0,1,0,0,1,1.0,0,-1.533345,1.0,-0.632361,1.008070,0.474941
136,-0.370651,0,0,1,0,0,1,1.0,1,0.786825,0.0,0.658564,-0.170908,0.133612
137,0.696583,0,0,1,0,0,1,1.0,1,1.554125,1.0,-0.551725,0.574668,0.197985


In [32]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [33]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.688798,0,0,1,0,0,1,1,1,-1.533345,0,0.893634,1.188930,1.218887
1,-0.688798,0,0,1,0,0,0,0,0,0.104884,1,-0.254678,-0.307894,-0.335276
2,-0.688798,0,0,1,0,0,0,0,1,-0.713198,1,0.595996,0.059450,0.228548
3,0.081262,1,0,0,0,1,1,0,1,0.940197,1,-0.296877,0.076321,-0.145668
4,1.273609,0,0,1,0,0,1,1,1,1.285347,0,-0.058378,0.786849,0.519421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.738387,1,0,0,0,1,0,0,0,1.192311,1,2.645651,-0.219083,0.751363
95,0.342484,0,0,0,0,1,0,0,1,3.094727,1,0.526028,-0.033134,0.168333
96,0.342484,0,0,1,0,0,1,1,1,-1.533345,1,-0.230255,0.867254,0.481447
97,-0.815082,0,0,1,0,0,1,1,0,-1.533345,0,0.706707,0.336049,0.486335


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [46]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:30:45,415] A new study created in memory with name: no-name-564c5fa8-92cb-4aa4-8bcd-58b5a937a3b4


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.5465116279069767
Fold 3 C-index: 0.7106382978723405


[I 2024-04-13 17:30:46,040] A new study created in memory with name: no-name-5058bd04-2023-4ca5-84af-c521f7950c6c


Fold 4 C-index: 0.6577946768060836
Fold 5 C-index: 0.592274678111588
[I 2024-04-13 17:30:46,017] Trial 0 finished with value: 0.6185753302429834 and parameters: {}. Best is trial 0 with value: 0.6185753302429834.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6185753302429834], datetime_start=datetime.datetime(2024, 4, 13, 17, 30, 45, 449124), datetime_complete=datetime.datetime(2024, 4, 13, 17, 30, 46, 17128), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6185753302429834


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2535303072491698
Fold 2 IBS: 0.2701123265297601
Fold 3 IBS: 0.21505701313669706
Fold 4 IBS: 0.27439723254313414
Fold 5 IBS: 0.22842205561435375
[I 2024-04-13 17:30:46,819] Trial 0 finished with value: 0.24830378701462297 and parameters: {}. Best is trial 0 with value: 0.24830378701462297.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.24830378701462297], datetime_start=datetime.datetime(2024, 4, 13, 17, 30, 46, 77347), datetime_complete=datetime.datetime(2024, 4, 13, 17, 30, 46, 819440), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.24830378701462297


In [47]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.619
train_ibs:  0.248


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

ValueError: search direction contains NaN or infinite values

In [52]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [53]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [54]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:30:59,275] A new study created in memory with name: no-name-8d1fb99d-8377-497c-8afd-0419723d3e1b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.603585657370518
Fold 2 C-index: 0.5058139534883721
Fold 3 C-index: 0.5617021276595745


[I 2024-04-13 17:30:59,738] A new study created in memory with name: no-name-aaaa2556-736c-42bc-8684-ad6d045c167a


Fold 4 C-index: 0.49809885931558934
Fold 5 C-index: 0.630901287553648
[I 2024-04-13 17:30:59,719] Trial 0 finished with value: 0.5600203770775404 and parameters: {}. Best is trial 0 with value: 0.5600203770775404.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5600203770775404], datetime_start=datetime.datetime(2024, 4, 13, 17, 30, 59, 320413), datetime_complete=datetime.datetime(2024, 4, 13, 17, 30, 59, 718793), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5600203770775404


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709728729422
Fold 2 IBS: 0.23203988447801246
Fold 3 IBS: 0.2289818653708319
Fold 4 IBS: 0.24197477735363535
Fold 5 IBS: 0.2293955872908494
[I 2024-04-13 17:31:00,372] Trial 0 finished with value: 0.23592784235612468 and parameters: {}. Best is trial 0 with value: 0.23592784235612468.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784235612468], datetime_start=datetime.datetime(2024, 4, 13, 17, 30, 59, 793254), datetime_complete=datetime.datetime(2024, 4, 13, 17, 31, 0, 372562), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784235612468


In [55]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [56]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.56
train_ibs:  0.236


#### Test

In [57]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [58]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.533


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [59]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [60]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:31:02,246] A new study created in memory with name: no-name-84346c83-93f3-4e8b-b106-143a424c44dc


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.562015503875969
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901


[I 2024-04-13 17:31:03,250] A new study created in memory with name: no-name-98241ac8-a62e-42eb-bd88-77d412b11dc3


Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:31:03,239] Trial 0 finished with value: 0.6303750707301188 and parameters: {}. Best is trial 0 with value: 0.6303750707301188.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6303750707301188], datetime_start=datetime.datetime(2024, 4, 13, 17, 31, 2, 334122), datetime_complete=datetime.datetime(2024, 4, 13, 17, 31, 3, 239222), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6303750707301188


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.25088041795120963
Fold 2 IBS: 0.2581733640424138
Fold 3 IBS: 0.2109028541228368
Fold 4 IBS: 0.2706797108128965
Fold 5 IBS: 0.22682720432557557
[I 2024-04-13 17:31:04,440] Trial 0 finished with value: 0.24349271025098645 and parameters: {}. Best is trial 0 with value: 0.24349271025098645.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.24349271025098645], datetime_start=datetime.datetime(2024, 4, 13, 17, 31, 3, 303823), datetime_complete=datetime.datetime(2024, 4, 13, 17, 31, 4, 439958), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.24349271025098645


In [61]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [62]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.63
train_ibs:  0.243


#### Test

In [63]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [64]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.571


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.279


In [65]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [66]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:31:06,475] A new study created in memory with name: no-name-037b0e20-3fce-494d-8881-cd3d01b01d63


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5542635658914729
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:31:07,348] Trial 0 finished with value: 0.6288246831332196 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6288246831332196.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5542635658914729
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:31:08,449] Trial 1 finished with value: 0.6288246831332196 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6288246831332196.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:31:09,406] Trial 2 finished with value: 0.6167875603066961 and parameters: {'l1_ratio': 0.22692876841884668}. 

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5542635658914729
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:31:37,508] Trial 24 finished with value: 0.6288246831332196 and parameters: {'l1_ratio': 0.7805647035680947}. Best is trial 6 with value: 0.6303750707301188.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.562015503875969
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:31:38,898] Trial 25 finished with value: 0.6303750707301188 and parameters: {'l1_ratio': 0.9368557715647121}. Best is trial 6 with value: 0.6303750707301188.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5019011406844106
Fold 5 C-index: 0.6523605150214592
[I 2024-04-13 17:31:39,655] Trial 26 finished with value: 0.5618472891287848 and parameters: {'l1_ratio': 0.015423757551295547}.

Fold 2 C-index: 0.5542635658914729
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:32:06,618] Trial 48 finished with value: 0.6288246831332196 and parameters: {'l1_ratio': 0.7768768486379835}. Best is trial 6 with value: 0.6303750707301188.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5542635658914729
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:32:07,583] Trial 49 finished with value: 0.6288246831332196 and parameters: {'l1_ratio': 0.8386679916089912}. Best is trial 6 with value: 0.6303750707301188.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.562015503875969
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.5965665236051502
[I 2024-04-13 17:32:08,478] Trial 50 finished with value: 0.6303750707301188 and parameters: {'l1_ratio': 0.945704027694845}. Best is trial 6 with value: 0.6303750

...

Fold 1 IBS: 0.2525836928556447
Fold 2 IBS: 0.2664404398563772
Fold 3 IBS: 0.21053233502960458
Fold 4 IBS: 0.2706211145594758
Fold 5 IBS: 0.22676170219108188
[I 2024-04-13 17:33:07,130] Trial 0 finished with value: 0.24538785689843684 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.24538785689843684.
Fold 1 IBS: 0.25253920525266704
Fold 2 IBS: 0.2662052808420901
Fold 3 IBS: 0.20996801490180272
Fold 4 IBS: 0.27074095333423953
Fold 5 IBS: 0.22672754258256947
[I 2024-04-13 17:33:08,400] Trial 1 finished with value: 0.2452361993826738 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.2452361993826738.
Fold 1 IBS: 0.25261539412518685
Fold 2 IBS: 0.2325812298093743
Fold 3 IBS: 0.20913119763580473
Fold 4 IBS: 0.27077104841709276
Fold 5 IBS: 0.22673601778495275
[I 2024-04-13 17:33:09,527] Trial 2 finished with value: 0.23836697755448227 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.2383669775544822

Fold 1 IBS: 0.252624043135885
Fold 2 IBS: 0.23255883455393458
Fold 3 IBS: 0.20893882883297735
Fold 4 IBS: 0.27074736350160367
Fold 5 IBS: 0.22669900945256133
[I 2024-04-13 17:33:33,043] Trial 25 finished with value: 0.23831361589539238 and parameters: {'l1_ratio': 0.21973436884995384}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.25246686300528276
Fold 2 IBS: 0.2661969101333031
Fold 3 IBS: 0.21068710549715425
Fold 4 IBS: 0.27069702623359165
Fold 5 IBS: 0.22669763552350894
[I 2024-04-13 17:33:34,500] Trial 26 finished with value: 0.24534910807856813 and parameters: {'l1_ratio': 0.3527001258697712}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.25278223742816774
Fold 2 IBS: 0.23229046211821905
Fold 3 IBS: 0.22661042410447052
Fold 4 IBS: 0.270855222018163
Fold 5 IBS: 0.226626089871234
[I 2024-04-13 17:33:35,508] Trial 27 finished with value: 0.24183288710805084 and parameters: {'l1_ratio': 0.13360026443673156}. Best is trial 24 with value: 0.2348120204

Fold 5 IBS: 0.22666605038281498
[I 2024-04-13 17:33:59,362] Trial 49 finished with value: 0.24178266297170428 and parameters: {'l1_ratio': 0.1518326539107156}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.25266240249412636
Fold 2 IBS: 0.23247632388876066
Fold 3 IBS: 0.20842985845168258
Fold 4 IBS: 0.2707546257420046
Fold 5 IBS: 0.22665258748020203
[I 2024-04-13 17:34:00,437] Trial 50 finished with value: 0.23819515961135526 and parameters: {'l1_ratio': 0.19340943531911486}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.24544954020658022
Fold 2 IBS: 0.2321572574037747
Fold 3 IBS: 0.2272316951999604
Fold 4 IBS: 0.24378652935635142
Fold 5 IBS: 0.22582386876850116
[I 2024-04-13 17:34:01,051] Trial 51 finished with value: 0.2348897781870336 and parameters: {'l1_ratio': 0.08640058638065554}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.24616826583690393
Fold 2 IBS: 0.2320652195968878
Fold 3 IBS: 0.22799004211711335
Fold 4 IBS: 0.2434204

Fold 1 IBS: 0.252713111132158
Fold 2 IBS: 0.23239323708181045
Fold 3 IBS: 0.20770103603701506
Fold 4 IBS: 0.2707378309292403
Fold 5 IBS: 0.22667740702580663
[I 2024-04-13 17:34:18,596] Trial 74 finished with value: 0.2380445244412061 and parameters: {'l1_ratio': 0.16695008087283925}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.2528601149267691
Fold 2 IBS: 0.23219255759525126
Fold 3 IBS: 0.22703843032528304
Fold 4 IBS: 0.24380222927085038
Fold 5 IBS: 0.22545003786977097
[I 2024-04-13 17:34:19,182] Trial 75 finished with value: 0.23626867399758494 and parameters: {'l1_ratio': 0.09973963884109015}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.25277974872052894
Fold 2 IBS: 0.23229330708700294
Fold 3 IBS: 0.22659963895619875
Fold 4 IBS: 0.2707619785171471
Fold 5 IBS: 0.22663261267114376
[I 2024-04-13 17:34:20,068] Trial 76 finished with value: 0.2418134571904043 and parameters: {'l1_ratio': 0.13454314551947133}. Best is trial 24 with value: 0.234812020

Fold 5 IBS: 0.22668719760261888
[I 2024-04-13 17:34:38,756] Trial 98 finished with value: 0.24178445061414738 and parameters: {'l1_ratio': 0.1549055492978006}. Best is trial 24 with value: 0.23481202042596405.
Fold 1 IBS: 0.2455139097004322
Fold 2 IBS: 0.2321453886883713
Fold 3 IBS: 0.22730388452260952
Fold 4 IBS: 0.24377320629239763
Fold 5 IBS: 0.2259645464095699
[I 2024-04-13 17:34:39,326] Trial 99 finished with value: 0.23494018712267611 and parameters: {'l1_ratio': 0.08168193521815892}. Best is trial 24 with value: 0.23481202042596405.


* Best trial for IBS: 
 FrozenTrial(number=24, state=TrialState.COMPLETE, values=[0.23481202042596405], datetime_start=datetime.datetime(2024, 4, 13, 17, 33, 30, 972928), datetime_complete=datetime.datetime(2024, 4, 13, 17, 33, 31, 566163), params={'l1_ratio': 0.09388196923015177}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=24, value=Non

In [67]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [68]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.63
train_ibs:  0.235


#### Test

In [69]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [70]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.980766121964777)

test_cindex : 0.572


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.09388196923015177)

test_ibs:  0.229


In [71]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [72]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:34:39,791] A new study created in memory with name: no-name-83fcad1d-88b4-4993-b722-a28710a37e62


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6201550387596899
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.6673003802281369
Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:34:47,352] Trial 0 finished with value: 0.6187977301006736 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6187977301006736.
Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6472868217054264
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.6159695817490495
Fold 5 C-index: 0.6094420600858369
[I 2024-04-13 17:34:52,698] Trial 1 finished with value: 0.6237488136710192 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 3 C-index: 0.776595744680851
Fold 4 C-index: 0.7262357414448669
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:36:05,542] Trial 15 finished with value: 0.6815364844139735 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 4, 'min_samples_leaf': 16, 'max_depth': 1, 'n_estimators': 32, 'oob_score': True, 'max_samples': 0.8219485202814404, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.20648604556011108, 'warm_start': True}. Best is trial 15 with value: 0.6815364844139735.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:36:06,167] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 5, 'min_samples_leaf': 16, 'max_depth': 1, 'n_estimators': 4, 'oob_score': True, 'max_samples': 0.44827686549014745, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.37909912926450273, 'warm_start': True}. Best is trial 15 with value: 0.6815364844139735.
Fold 1 C-index: 0

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.7829787234042553
Fold 4 C-index: 0.7129277566539924
Fold 5 C-index: 0.6824034334763949
[I 2024-04-13 17:36:48,186] Trial 30 finished with value: 0.6883010388853158 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 13, 'n_estimators': 394, 'oob_score': True, 'max_samples': 0.6919164029113454, 'max_features': None, 'min_weight_fraction_leaf': 0.18416154592106396, 'warm_start': True}. Best is trial 24 with value: 0.7028777296239799.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.7957446808510639
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6952789699570815
[I 2024-04-13 17:36:54,686] Trial 31 finished with value: 0.6991838537239718 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 19, 'min_samples_leaf': 14, 'max_depth': 15, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.742540777440605

...

Fold 1 IBS: 0.2370347091364531
Fold 2 IBS: 0.21865616933268361
Fold 3 IBS: 0.2136904590169095
Fold 4 IBS: 0.2330616724900941
Fold 5 IBS: 0.21522492222501335
[I 2024-04-13 17:45:01,746] Trial 32 finished with value: 0.22353358644023075 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 310, 'oob_score': True, 'max_samples': 0.8040571874379057, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.18786523282060735}. Best is trial 32 with value: 0.22353358644023075.
Fold 1 IBS: 0.2369596498960518
Fold 2 IBS: 0.21919381104819285
Fold 3 IBS: 0.21367435784125732
Fold 4 IBS: 0.2331924481484345
Fold 5 IBS: 0.21510595787220624
[I 2024-04-13 17:45:10,702] Trial 33 finished with value: 0.22362524496122854 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 309, 'oob_score': True, 'max_samples': 0.7910445247744765, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 5 IBS: 0.21452585999296125
[I 2024-04-13 17:48:30,337] Trial 47 finished with value: 0.22331758578489697 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 332, 'oob_score': True, 'max_samples': 0.8648447148686852, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19908924320632904}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23571677073756844
Fold 2 IBS: 0.21862106487993588
Fold 3 IBS: 0.21456356649548577
Fold 4 IBS: 0.23300957311771886
Fold 5 IBS: 0.2147097776323396
[I 2024-04-13 17:48:43,141] Trial 48 finished with value: 0.2233241505726097 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 9, 'n_estimators': 339, 'oob_score': True, 'max_samples': 0.85048001681846, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.21469064698337334}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23559067856645097
Fold 2 IBS: 0.218738

Fold 1 IBS: 0.24251844789708726
Fold 2 IBS: 0.2210411248095999
Fold 3 IBS: 0.2174884582534811
Fold 4 IBS: 0.23327796476633547
Fold 5 IBS: 0.21325865662478627
[I 2024-04-13 17:52:06,469] Trial 63 finished with value: 0.225516930470258 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 14, 'min_samples_leaf': 5, 'max_depth': 11, 'n_estimators': 440, 'oob_score': True, 'max_samples': 0.8434390256503708, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.13262761517233745}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.24154578994370174
Fold 2 IBS: 0.22210454661611326
Fold 3 IBS: 0.21614753754341642
Fold 4 IBS: 0.23303912664616094
Fold 5 IBS: 0.21337979688122055
[I 2024-04-13 17:52:22,546] Trial 64 finished with value: 0.2252433595261226 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 421, 'oob_score': True, 'max_samples': 0.9071936553420485, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.21423498837277127
[I 2024-04-13 17:55:59,392] Trial 78 finished with value: 0.22373591837101642 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 10, 'n_estimators': 346, 'oob_score': True, 'max_samples': 0.8854068847594949, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.23765891161800223}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23570867974207424
Fold 2 IBS: 0.22060193373952594
Fold 3 IBS: 0.2144464193518562
Fold 4 IBS: 0.23476296957407278
Fold 5 IBS: 0.21578644913062256
[I 2024-04-13 17:56:13,301] Trial 79 finished with value: 0.22426129030763034 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 291, 'oob_score': True, 'max_samples': 0.9991782770608287, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2796131771286568}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.25860616056552466
Fold 2 IBS: 0.247

Fold 1 IBS: 0.2364707434109151
Fold 2 IBS: 0.21910149125141531
Fold 3 IBS: 0.21420284080337593
Fold 4 IBS: 0.23438012728527638
Fold 5 IBS: 0.21479897289302796
[I 2024-04-13 18:01:42,437] Trial 94 finished with value: 0.22379083512880213 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 6, 'n_estimators': 430, 'oob_score': True, 'max_samples': 0.892312681224388, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2167154554420358}. Best is trial 84 with value: 0.22302339534037285.
Fold 1 IBS: 0.23950861114017571
Fold 2 IBS: 0.22006606579763324
Fold 3 IBS: 0.2159169717512573
Fold 4 IBS: 0.2336883258166932
Fold 5 IBS: 0.21371153072549798
[I 2024-04-13 18:02:05,824] Trial 95 finished with value: 0.22457830104625148 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 481, 'oob_score': True, 'max_samples': 0.8276360496389354, 'max_features': 'log2', 'min_weight_fraction_leaf'

In [73]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [74]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.76
train_ibs:  0.223


#### Test

In [75]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [76]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=5, max_leaf_nodes=8,
                     max_samples=0.9313711932374781, min_samples_split=12,
                     min_weight_fraction_leaf=0.011633867937315842,
                     n_estimators=214, random_state=123, warm_start=True)

test_cindex:  0.633


RandomSurvivalForest(max_depth=6, max_features='log2', max_leaf_nodes=20,
                     max_samples=0.8096737689830545, min_samples_leaf=2,
                     min_samples_split=11,
                     min_weight_fraction_leaf=0.19628098798162147,
                     n_estimators=453, oob_score=True, random_state=123)

test_ibs:  0.216


In [77]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [78]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [79]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 18:03:25,382] A new study created in memory with name: no-name-4ee4f212-f8a7-44b5-b1dd-4ef250669870


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.7489361702127659
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.6695278969957081
[I 2024-04-13 18:03:27,804] Trial 0 finished with value: 0.6810141068632278 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6810141068632278.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:03:34,198] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.6627906976744186
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.6825095057034221
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 18:05:01,924] Trial 16 finished with value: 0.6718042314769685 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.6867027610976928.
Fold 1 C-index: 0.6115537848605578
Fold 2 C-index: 0.6453488372093024
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.6825095057034221
Fold 5 C-index: 0.6566523605150214
[I 2024-04-13 18:05:04,711] Trial 17 finished with value: 0.6672980040406395 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.7682403433476395
[I 2024-04-13 18:06:03,860] Trial 31 finished with value: 0.7161536582914563 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 207, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.9402340730195752, 'min_weight_fraction_leaf': 0.07589392630375186}. Best is trial 28 with value: 0.7175262582650991.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.7702127659574468
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.721030042918455
[I 2024-04-13 18:06:07,394] Trial 32 finished with value: 0.6961121726190391 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 207, 'oob_score': False, 'warm_start': True, 'max_features':

...

Fold 1 IBS: 0.24654710532719232
Fold 2 IBS: 0.23226075421028072
Fold 3 IBS: 0.22941657391206893
Fold 4 IBS: 0.24156072638580853
Fold 5 IBS: 0.23017807225897458
[I 2024-04-13 18:15:13,493] Trial 30 finished with value: 0.235992646418865 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 12, 'max_depth': 15, 'n_estimators': 225, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.46244718326068934, 'min_weight_fraction_leaf': 0.23846812660748434}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23751141455085345
Fold 2 IBS: 0.2151581315756588
Fold 3 IBS: 0.20565940925991258
Fold 4 IBS: 0.22542469872025206
Fold 5 IBS: 0.21030193870732267
[I 2024-04-13 18:15:28,140] Trial 31 finished with value: 0.2188111185627999 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 500, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.5

Fold 1 IBS: 0.23579057506686668
Fold 2 IBS: 0.21781214726067985
Fold 3 IBS: 0.2066254011517932
Fold 4 IBS: 0.22753811203645144
Fold 5 IBS: 0.2131782901601682
[I 2024-04-13 18:18:24,406] Trial 45 finished with value: 0.22018890513519188 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 5, 'n_estimators': 367, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.7526038702156265, 'min_weight_fraction_leaf': 0.19723066794982302}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23864304620933718
Fold 2 IBS: 0.2216376067444548
Fold 3 IBS: 0.21798786862601383
Fold 4 IBS: 0.2317000936375164
Fold 5 IBS: 0.21855619191514636
[I 2024-04-13 18:18:41,842] Trial 46 finished with value: 0.2257049614264937 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 306, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8402191

Fold 1 IBS: 0.23905204688392212
Fold 2 IBS: 0.2187928517163343
Fold 3 IBS: 0.2102452041922602
Fold 4 IBS: 0.22890896945323644
Fold 5 IBS: 0.21433456345556318
[I 2024-04-13 18:20:56,190] Trial 60 finished with value: 0.22226672714026327 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 11, 'max_depth': 16, 'n_estimators': 74, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9528336631358474, 'min_weight_fraction_leaf': 0.23484468684942653}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23541239037810044
Fold 2 IBS: 0.2157600681640186
Fold 3 IBS: 0.20665073702737224
Fold 4 IBS: 0.22705259747647938
Fold 5 IBS: 0.2114858383168534
[I 2024-04-13 18:21:09,766] Trial 61 finished with value: 0.21927232627256482 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 8, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 397, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6

Fold 1 IBS: 0.23749669116970254
Fold 2 IBS: 0.21417891364052918
Fold 3 IBS: 0.2066472918191204
Fold 4 IBS: 0.22577332010612122
Fold 5 IBS: 0.20791194308464916
[I 2024-04-13 18:23:57,508] Trial 75 finished with value: 0.2184016319640245 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 6, 'n_estimators': 346, 'oob_score': True, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6413926717278569, 'min_weight_fraction_leaf': 0.0007388507314896667}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23958121408927258
Fold 2 IBS: 0.21329051656876732
Fold 3 IBS: 0.20685348469946407
Fold 4 IBS: 0.2243628744610485
Fold 5 IBS: 0.2076885994354418
[I 2024-04-13 18:24:22,115] Trial 76 finished with value: 0.21835533785079883 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 356, 'oob_score': True, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6

Fold 1 IBS: 0.23843719064657817
Fold 2 IBS: 0.22064066243477068
Fold 3 IBS: 0.216117076423818
Fold 4 IBS: 0.23057174337958733
Fold 5 IBS: 0.21778600564999445
[I 2024-04-13 18:27:49,269] Trial 90 finished with value: 0.22471053570694974 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 274, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.42143690848778903, 'min_weight_fraction_leaf': 0.05305541913345835}. Best is trial 83 with value: 0.21749549175321342.
Fold 1 IBS: 0.23636584853089593
Fold 2 IBS: 0.2160409129934387
Fold 3 IBS: 0.20518796152371307
Fold 4 IBS: 0.22686151762465062
Fold 5 IBS: 0.21138354158709388
[I 2024-04-13 18:28:00,342] Trial 91 finished with value: 0.21916795645195847 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 17, 'min_samples_leaf': 4, 'max_depth': 4, 'n_estimators': 307, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.378

In [80]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [81]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.217


#### Test

In [82]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [83]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=13, max_features=None, max_leaf_nodes=16,
                   max_samples=0.8020091028727125, min_samples_leaf=1,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.002435943155287043,
                   n_estimators=277, random_state=123, warm_start=True)

C-index score: 0.584


ExtraSurvivalTrees(max_depth=6, max_features='auto', max_leaf_nodes=16,
                   max_samples=0.44903497421134486, min_samples_leaf=4,
                   min_samples_split=9,
                   min_weight_fraction_leaf=0.018023565445760347,
                   n_estimators=351, oob_score=True, random_state=123)

IBS: 0.223


In [84]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [85]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 18:29:28,226] A new study created in memory with name: no-name-6ba7c145-2849-4b24-9b7a-a8cadeb49aa0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:30:30,370] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:31:08,521] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:47:07,806] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6059172111077464.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:49:06,380] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 19:10:08,102] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.6190116704947725.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 19:12:40,567] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:28:07,569] Trial 44 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9479336661285427, 'learning_rate': 0.0149361535238727, 'dropout_rate': 0.25335630680617827, 'n_estimators': 238, 'criterion': 'squared_error', 'ccp_alpha': 0.5490150526266808, 'min_weight_fraction_leaf': 0.4126953497238681, 'max_features': 'auto', 'min_impurity_decrease': 0.00078866474186123, 'validation_fraction': 0.9438667820963776, 'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 3}. Best is trial 42 with value: 0.2339713101479977.
Fold 1 IBS: 0.24678393645753394
Fold 2 IBS: 0.23045362112354736
Fold 3 IBS: 0.22798640772703402
Fold 4 IBS: 0.24133177588235574
Fold 5 IBS: 0.22819468131537443
[I 2024-04-13 21:28:19,968] Trial 45 finished with value: 0.2349500845011691 and parameters: {'subsample': 0.5852424762732177, 'learning_rate': 0.065638506610498

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:29:55,213] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.804828042321909, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.23926982708415356, 'n_estimators': 162, 'criterion': 'squared_error', 'ccp_alpha': 0.40074886285106287, 'min_weight_fraction_leaf': 0.2779066644542128, 'max_features': 'sqrt', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 19, 'max_depth': 4}. Best is trial 42 with value: 0.2339713101479977.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:30:19,051] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6106355356441384, 'learning_rate': 0.0175381503002972, 'dropout_rate': 0.184039934

Fold 5 IBS: 0.22939559304809248
[I 2024-04-13 21:32:28,093] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7279386202897999, 'learning_rate': 0.09984676062907398, 'dropout_rate': 0.12777796849617917, 'n_estimators': 154, 'criterion': 'squared_error', 'ccp_alpha': 1.4111498627316026, 'min_weight_fraction_leaf': 0.23517339076530247, 'max_features': 1, 'min_impurity_decrease': 0.00014974488025406402, 'validation_fraction': 0.6772907402354433, 'min_samples_split': 8, 'max_leaf_nodes': 16, 'min_samples_leaf': 20, 'max_depth': 3}. Best is trial 65 with value: 0.2330518973544506.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:32:30,499] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8919877355885673, 'learning_rate': 0.08526801927485488, 'dropout_rate': 0.1968183597819077, 'n_estimators': 45, 'cri

Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.2419747714592711
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:35:29,990] Trial 78 finished with value: 0.2359278435123307 and parameters: {'subsample': 0.9982285661999424, 'learning_rate': 0.09191504218778693, 'dropout_rate': 0.15698726385072695, 'n_estimators': 197, 'criterion': 'squared_error', 'ccp_alpha': 0.24807516981478814, 'min_weight_fraction_leaf': 0.20156232220196313, 'max_features': 0.1, 'min_impurity_decrease': 1.657879625668623e-05, 'validation_fraction': 0.743279291715464, 'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 11}. Best is trial 75 with value: 0.23239681107011897.
Fold 1 IBS: 0.24690614034739003
Fold 2 IBS: 0.23156150181893284
Fold 3 IBS: 0.22844484931909595
Fold 4 IBS: 0.24167312234765576
Fold 5 IBS: 0.22876368497640898
[I 2024-04-13 21:35:33,047] Trial 79 finished with value: 0.23546985976189672 and parameters: {'s

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:37:07,197] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9483409389571185, 'learning_rate': 0.08684995291290751, 'dropout_rate': 0.261199231115212, 'n_estimators': 33, 'criterion': 'squared_error', 'ccp_alpha': 0.262026471967215, 'min_weight_fraction_leaf': 0.30628737845191195, 'max_features': 0.1, 'min_impurity_decrease': 6.315494653924259e-06, 'validation_fraction': 0.7881463591488752, 'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 15, 'max_depth': 4}. Best is trial 75 with value: 0.23239681107011897.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-13 21:37:17,412] Trial 90 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9683729112775776, 'learning_rate': 0.0901748280920096

In [86]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [87]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.642
train_ibs:  0.232


#### Test

In [88]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [89]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0019473494055538935,
                                 criterion='squared_error',
                                 dropout_rate=0.43613571687055647,
                                 learning_rate=0.09542136158814303,
                                 max_depth=20, max_features='sqrt',
                                 max_leaf_nodes=20,
                                 min_impurity_decrease=5.255459802548745e-07,
                                 min_samples_leaf=15, min_samples_split=10,
                                 min_weight_fraction_leaf=0.36743918461462793,
                                 n_estimators=66, random_state=123,
                                 subsample=0.569175138136116,
                                 validation_fraction=0.7087056989518657)

C-index score: 0.669


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008911991304216094,
                                 criterion='squared_error',
                                 dropout_rate=0.1333613034046263,
                                 learning_rate=0.09420039979456486,
                                 max_depth=14, max_features=0.1,
                                 max_leaf_nodes=19,
                                 min_impurity_decrease=2.4912576060277585e-06,
                                 min_samples_leaf=17, min_samples_split=7,
                                 min_weight_fraction_leaf=0.2444025476799629,
                                 n_estimators=141, random_state=123,
                                 subsample=0.9998829056403309,
                                 validation_fraction=0.6909543967814417)

IBS: 0.224


In [90]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [91]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [92]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 21:38:07,710] A new study created in memory with name: no-name-cbb1b423-0a8d-471c-82c8-73246b0d72e2


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.6482889733840305
Fold 5 C-index: 0.6201716738197425
[I 2024-04-13 21:38:09,659] Trial 0 finished with value: 0.5866142624492122 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5866142624492122.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.6244635193133047
[I 2024-04-13 21:38:23,849] Trial 1 finished with value: 0.576446015578343 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.5866142624492122.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.5406976744186046
Fold 3 C-index: 0.574468085106383
Fold 4 C-index: 0.596958174904943
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 21:40:33,205] Trial 19 finished with value: 0.5937662023400497 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.6048986065729117.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5271317829457365
Fold 3 C-index: 0.5659574468085107
Fold 4 C-index: 0.6045627376425855
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 21:40:47,113] Trial 20 finished with value: 0.5909949216328471 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.6048986065729117.
Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.5387596899224806
Fold 3 C-index: 0.5829787234042553
F

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5680851063829787
Fold 4 C-index: 0.596958174904943
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 21:42:37,840] Trial 38 finished with value: 0.5855527998084764 and parameters: {'subsample': 0.3139010416420267, 'dropout_rate': 0.297543464189074, 'n_estimators': 75, 'learning_rate': 0.07586972024157643}. Best is trial 34 with value: 0.6200383792472979.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.548936170212766
Fold 4 C-index: 0.596958174904943
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 21:42:41,255] Trial 39 finished with value: 0.57893416795292 and parameters: {'subsample': 0.41872364564821285, 'dropout_rate': 0.5267513165392328, 'n_estimators': 159, 'learning_rate': 0.06892741183938003}. Best is trial 34 with value: 0.6200383792472979.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-

...

Fold 3 IBS: 0.23449904393194607
Fold 4 IBS: 0.243477517914051
Fold 5 IBS: 0.20898654378339646
[I 2024-04-13 22:08:05,334] Trial 19 finished with value: 0.23335008775341448 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 7 with value: 0.23106060701964246.
Fold 1 IBS: 0.244476360186189
Fold 2 IBS: 0.2305472199436817
Fold 3 IBS: 0.22377140228384404
Fold 4 IBS: 0.2386766254699274
Fold 5 IBS: 0.22296335334244682
[I 2024-04-13 22:08:06,287] Trial 20 finished with value: 0.23208699224521778 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 7 with value: 0.23106060701964246.
Fold 1 IBS: 0.24513654608372853
Fold 2 IBS: 0.23094110628575634
Fold 3 IBS: 0.22464827063854145
Fold 4 IBS: 0.23926825072734292
Fold 5 IBS: 0.22475950483168106
[I 2024-04-13 22:08:07,060] Trial 21 finish

Fold 3 IBS: 0.31041839361900425
Fold 4 IBS: 0.38439222893604497
Fold 5 IBS: 0.32738236379645497
[I 2024-04-13 22:08:34,730] Trial 38 finished with value: 0.3526236292421712 and parameters: {'subsample': 0.3278854808520768, 'dropout_rate': 0.13272164755980653, 'n_estimators': 381, 'learning_rate': 0.06814074928034364}. Best is trial 7 with value: 0.23106060701964246.
Fold 1 IBS: 0.29827692610119566
Fold 2 IBS: 0.282045562042545
Fold 3 IBS: 0.28301041532740556
Fold 4 IBS: 0.26637593036008284
Fold 5 IBS: 0.2779967799215671
[I 2024-04-13 22:08:36,613] Trial 39 finished with value: 0.2815411227505592 and parameters: {'subsample': 0.7834569849171229, 'dropout_rate': 0.9408456886999991, 'n_estimators': 179, 'learning_rate': 0.04617617206708862}. Best is trial 7 with value: 0.23106060701964246.
Fold 1 IBS: 0.324806407871569
Fold 2 IBS: 0.37786223186541407
Fold 3 IBS: 0.31482812552937167
Fold 4 IBS: 0.3227925487415433
Fold 5 IBS: 0.33867925222488654
[I 2024-04-13 22:08:45,435] Trial 40 finished

Fold 4 IBS: 0.23784017371297209
Fold 5 IBS: 0.21485810066702843
[I 2024-04-13 22:10:53,952] Trial 57 finished with value: 0.23097303215956835 and parameters: {'subsample': 0.455497864682559, 'dropout_rate': 0.727615869586519, 'n_estimators': 198, 'learning_rate': 0.010247024954642935}. Best is trial 55 with value: 0.23096923556657606.
Fold 1 IBS: 0.24352828586762612
Fold 2 IBS: 0.2308324022950569
Fold 3 IBS: 0.22450011836087874
Fold 4 IBS: 0.23758666795918215
Fold 5 IBS: 0.217482896231864
[I 2024-04-13 22:10:56,345] Trial 58 finished with value: 0.23078607414292157 and parameters: {'subsample': 0.4626553192113092, 'dropout_rate': 0.7303672304395288, 'n_estimators': 204, 'learning_rate': 0.007862561293053814}. Best is trial 58 with value: 0.23078607414292157.
Fold 1 IBS: 0.24417328945183361
Fold 2 IBS: 0.23043766192154386
Fold 3 IBS: 0.22362240990011897
Fold 4 IBS: 0.23864400560218796
Fold 5 IBS: 0.22198895903862992
[I 2024-04-13 22:10:58,889] Trial 59 finished with value: 0.23177326518

Fold 4 IBS: 0.24011681767071918
Fold 5 IBS: 0.2180561497748414
[I 2024-04-13 22:12:45,001] Trial 76 finished with value: 0.23105566720527654 and parameters: {'subsample': 0.3447701607161631, 'dropout_rate': 0.44910581572830965, 'n_estimators': 235, 'learning_rate': 0.006014704236887859}. Best is trial 67 with value: 0.2305246765527135.
Fold 1 IBS: 0.24517753021026703
Fold 2 IBS: 0.23065498965579573
Fold 3 IBS: 0.22488935552274253
Fold 4 IBS: 0.24047333252997696
Fold 5 IBS: 0.22503321076794347
[I 2024-04-13 22:12:49,832] Trial 77 finished with value: 0.23324568373734517 and parameters: {'subsample': 0.37306059300005784, 'dropout_rate': 0.48805498613846665, 'n_estimators': 303, 'learning_rate': 0.0017196299707487941}. Best is trial 67 with value: 0.2305246765527135.
Fold 1 IBS: 0.2440642480926844
Fold 2 IBS: 0.23788853785096567
Fold 3 IBS: 0.23708487980427928
Fold 4 IBS: 0.24926958370506158
Fold 5 IBS: 0.2050083932524473
[I 2024-04-13 22:12:53,715] Trial 78 finished with value: 0.2346631

Fold 4 IBS: 0.240096804175716
Fold 5 IBS: 0.21220295505386813
[I 2024-04-13 22:19:18,632] Trial 95 finished with value: 0.23117808952996688 and parameters: {'subsample': 0.3683350446497741, 'dropout_rate': 0.6195112110508696, 'n_estimators': 256, 'learning_rate': 0.009118931904905743}. Best is trial 67 with value: 0.2305246765527135.
Fold 1 IBS: 0.24257214724086992
Fold 2 IBS: 0.2343278228700067
Fold 3 IBS: 0.23143612426238616
Fold 4 IBS: 0.24257936688981685
Fold 5 IBS: 0.20909399543337767
[I 2024-04-13 22:19:21,579] Trial 96 finished with value: 0.23200189133929144 and parameters: {'subsample': 0.308291704410664, 'dropout_rate': 0.6432020013429081, 'n_estimators': 243, 'learning_rate': 0.011917866704154519}. Best is trial 67 with value: 0.2305246765527135.
Fold 1 IBS: 0.24444085952060532
Fold 2 IBS: 0.2300912445266555
Fold 3 IBS: 0.2240606885107319
Fold 4 IBS: 0.24047891143824324
Fold 5 IBS: 0.22366696464055894
[I 2024-04-13 22:19:24,356] Trial 97 finished with value: 0.23254773372735

In [93]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [94]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.62
train_ibs:  0.231


#### Test

In [95]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [96]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.644905787703748,
                                              learning_rate=0.09966139122330783,
                                              n_estimators=5, random_state=123,
                                              subsample=0.14154027296706004)

C-index score: 0.543


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.604986146999221,
                                              learning_rate=0.006338930484056821,
                                              n_estimators=310,
                                              random_state=123,
                                              subsample=0.35814936289595134)

IBS: 0.235


In [97]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [98]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.791,1.0
Randomsurvivalforest,0.760,2.0
GradientBoosting,0.642,3.0
CoxLasso,0.630,4.5
CoxElastic,0.630,4.5
ComponentwiseGradientBoosting,0.620,6.0
CoxPH,0.619,7.0
CoxRidge,0.560,8.0


In [99]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.217,1.0
Randomsurvivalforest,0.223,2.0
ComponentwiseGradientBoosting,0.231,3.0
GradientBoosting,0.232,4.0
CoxElastic,0.235,5.0
CoxRidge,0.236,6.0
CoxLasso,0.243,7.0
CoxPH,0.248,8.0


In [100]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.669,1.0
Randomsurvivalforest,0.633,2.0
ExtraSurvivalTrees,0.584,3.0
CoxElastic,0.572,4.0
CoxLasso,0.571,5.0
ComponentwiseGradientBoosting,0.543,6.0
CoxRidge,0.533,7.0


In [101]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.216,1.0
ExtraSurvivalTrees,0.223,2.0
GradientBoosting,0.224,3.0
CoxRidge,0.229,4.5
CoxElastic,0.229,4.5
ComponentwiseGradientBoosting,0.235,6.0
CoxLasso,0.279,7.0


In [102]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/yeojohnson/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_yeojohnson_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [103]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-13
